# Silver to Gold Transformation

## Import Libraries

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 3, Finished, Available, Finished, False)

## Read Silver Tables

In [2]:
silver_customers = spark.table("silver_customers")
silver_campaigns = spark.table("silver_campaigns")
silver_customer_events = spark.table("silver_customer_events")
silver_products = spark.table("silver_products")
silver_support_cases = spark.table("silver_support_cases")

print("Silver tables loaded successfully")

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 4, Finished, Available, Finished, False)

Silver tables loaded successfully


In [3]:
display(silver_customers)

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0ca30a7-830e-4d98-96e2-fdd15b649677)

## Build Gold Layer

#### Create Gold Customer Engagement

In [4]:
customer_event_summary = (
    silver_customer_events
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_events"),
        max("event_timestamp").alias("last_activity")
    )
)

display(customer_event_summary)

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 63762ae0-3ce8-4c11-a1fc-96c914b59df5)

In [5]:
support_summary = (
    silver_support_cases
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_tickets"),
        sum(when(col("status")=="Open",1).otherwise(0)).alias("open_tickets"),
        sum(when(col("status")=="Resolved",1).otherwise(0)).alias("resolved_tickets")
    )
)

display(support_summary)

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5435d045-839d-4e8d-b068-56017765663e)

In [6]:
gold_customer_engagement = (
    silver_customers
    .join(customer_event_summary, "customer_id", "left")
    .join(support_summary, "customer_id", "left")
    .fillna(0, subset=[
        "total_events",
        "total_tickets",
        "open_tickets",
        "resolved_tickets"
    ])
)

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 8, Finished, Available, Finished, False)

In [7]:
gold_customer_engagement = (
    gold_customer_engagement
    .withColumn(
        "engagement_level",
        when(col("total_events") >= 15, "High")
        .when(col("total_events") >= 8, "Medium")
        .otherwise("Low")
    )
)

display(gold_customer_engagement)

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a667153-bbd6-4f57-9342-a85465358a49)

### Save gold_customer_engagement

In [8]:
gold_customer_engagement.write.mode("overwrite").format("delta").saveAsTable("gold_customer_engagement")

print("gold_customer_engagement table created successfully")

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 10, Finished, Available, Finished, False)

gold_customer_engagement table created successfully


### Create gold_campaign_performance

In [9]:
campaign_event_summary = (
    silver_customer_events
    .filter(col("campaign_id").isNotNull())
    .filter(col("campaign_id") != "")
    .groupBy("campaign_id")
    .agg(
        count("*").alias("total_campaign_events"),
        countDistinct("customer_id").alias("unique_customers_engaged"),
        sum(when(col("event_type") == "campaign_click", 1).otherwise(0)).alias("campaign_clicks"),
        sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
        sum(when(col("event_type") == "purchase", col("event_value")).otherwise(0)).alias("campaign_revenue")
    )
)

gold_campaign_performance = (
    silver_campaigns
    .join(campaign_event_summary, "campaign_id", "left")
    .fillna(0, subset=[
        "total_campaign_events",
        "unique_customers_engaged",
        "campaign_clicks",
        "purchases",
        "campaign_revenue"
    ])
    .withColumn(
        "conversion_rate",
        when(col("campaign_clicks") > 0, round((col("purchases") / col("campaign_clicks")) * 100, 2))
        .otherwise(0)
    )
)

display(gold_campaign_performance)

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df2f8116-f77d-4bce-b301-75f230819bcb)

### Save gold_campaign_performance

In [10]:
gold_campaign_performance.write.mode("overwrite").format("delta").saveAsTable("gold_campaign_performance")

print("gold_campaign_performance table created successfully")

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 12, Finished, Available, Finished, False)

gold_campaign_performance table created successfully


### Create gold_support_summary

In [11]:
gold_support_summary = (
    silver_support_cases
    .withColumn(
        "resolution_days",
        datediff(col("resolved_date"), col("created_date"))
    )
    .groupBy("issue_type", "priority")
    .agg(
        count("*").alias("total_tickets"),
        sum(when(col("status") == "Open", 1).otherwise(0)).alias("open_tickets"),
        sum(when(col("status") == "Resolved", 1).otherwise(0)).alias("resolved_tickets"),
        sum(when(col("status") == "Closed", 1).otherwise(0)).alias("closed_tickets"),
        round(avg("resolution_days"), 2).alias("avg_resolution_days")
    )
)

display(gold_support_summary)

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2524efbb-b1e9-4f67-a216-47b49c5c8695)

### Save gold_support_summary

In [12]:
gold_support_summary.write.mode("overwrite").format("delta").saveAsTable("gold_support_summary")

print("gold_support_summary table created successfully")

StatementMeta(, 732dc42d-bde8-409a-8034-adc97df5cc81, 14, Finished, Available, Finished, False)

gold_support_summary table created successfully
